In [1]:
import torch
import torch.nn as nn

# 1. 일반적인 Convolution 블록 (Standard Conv)
class StandardBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(StandardBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

# 2. MobileNet의 핵심 블록 (Depthwise Separable Conv)
class MobileNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(MobileNetBlock, self).__init__()
        self.depsep = nn.Sequential(
            # (1) Depthwise Convolution: 채널별로 따로 연산 (groups=in_channels가 핵심!)
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            
            # (2) Pointwise Convolution: 1x1 Conv로 채널 간 정보 통합
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.depsep(x)
    
if __name__ == "__main__":
    # 가상의 입력 데이터 (Batch=1, Channel=32, H=64, W=64)
    x = torch.randn(1, 32, 64, 64)
    
    std_block = StandardBlock(32, 64)
    mb_block = MobileNetBlock(32, 64)
    
    print(f"Input Shape: {x.shape}")
    print(f"Standard Block Output: {std_block(x).shape}")
    print(f"MobileNet Block Output: {mb_block(x).shape}")
    
    # 파라미터 수 비교 (여기서 차이가 확 드러납니다!)
    std_params = sum(p.numel() for p in std_block.parameters())
    mb_params = sum(p.numel() for p in mb_block.parameters())
    
    print("-" * 30)
    print(f"Standard Block Params: {std_params}")
    print(f"MobileNet Block Params: {mb_params}")
    print(f"Reduction Ratio: {std_params / mb_params:.2f}x smaller!")

Input Shape: torch.Size([1, 32, 64, 64])
Standard Block Output: torch.Size([1, 64, 64, 64])
MobileNet Block Output: torch.Size([1, 64, 64, 64])
------------------------------
Standard Block Params: 18560
MobileNet Block Params: 2528
Reduction Ratio: 7.34x smaller!
